In this code, 
1. first the POT of hourly variable and the associated thresholds of the POT for each year are achieved. 
3. Based on POT variable (center variable), a 2.5 days before and after each POT time is considered to find the max of other variables (that can be hourly or daily).

In [4]:
import pandas as pd
import numpy as np
import glob
from datetime import timedelta
import os
from pathlib import Path

In [ ]:
def POT_Hourly_all(center_variable_time, associated_hourly_variable_time, 
                   center_variable_input_column, associated_hourly_variable_input_column, 
                   center_variable_input_path, associated_hourly_variable_input_path, 
                   center_variable_output_column, associated_hourly_variable_output_column, 
                   number_of_events_per_year, thresholds_output_path, pot_output_path, stationary=True):

    """
    Extracts POT events from an hourly center variable and identifies the associated
    maximum values of TWO hourly variables within a ±5-day event window.

    positive lag = associated variable peaks after the center variable
    negative lag = associated variable peaks before the center variable
    """

    import pandas as pd
    import numpy as np
    from datetime import timedelta

    # ─────────────────────────────────────────────────────────────
    # Read datasets
    # ─────────────────────────────────────────────────────────────
    df_center_variable = pd.read_csv(
        center_variable_input_path,
        parse_dates=[center_variable_time]
    )
    df_center_variable["time"] = pd.to_datetime(df_center_variable[center_variable_time])
    df_center_variable = df_center_variable.set_index("time").sort_index()

    df_associated_hourly_variable = pd.read_csv(
        associated_hourly_variable_input_path,
        parse_dates=[associated_hourly_variable_time]
    )
    df_associated_hourly_variable["time"] = pd.to_datetime(
        df_associated_hourly_variable[associated_hourly_variable_time]
    )
    df_associated_hourly_variable = df_associated_hourly_variable.set_index("time").sort_index()

    # ─────────────────────────────────────────────────────────────
    # Parameters
    # ─────────────────────────────────────────────────────────────
    center_variable_series = df_center_variable[center_variable_input_column].dropna()

    decluster_window = timedelta(days=2.5)
    associated_variable_decluster_window = timedelta(days=2)

    all_pot = []
    thresholds = []

    # ─────────────────────────────────────────────────────────────
    # Helper function: find associated hourly maximum and lag
    # ─────────────────────────────────────────────────────────────
    def get_hourly_associated_max(df, column, event_time, window):
        start = event_time - window
        end = event_time + window

        window_data = df.loc[start:end]
        s = window_data[column].dropna()

        if s.empty:
            return np.nan

        max_time = s.idxmax()
        max_value = s.loc[max_time]

        return max_value

    # ─────────────────────────────────────────────────────────────
    # Stationary POT threshold
    # ─────────────────────────────────────────────────────────────
    if stationary:

        center_variable_series_sorted = center_variable_series.sort_values(ascending=False)

        selected = []
        selected_values = []

        total_years = center_variable_series.index.year.nunique()
        total_events = number_of_events_per_year * total_years

        for idx, val in center_variable_series_sorted.items():

            if any(abs(idx - sel) <= decluster_window for sel in selected):
                continue

            selected.append(idx)
            selected_values.append(val)

            if len(selected) == total_events:
                break

        if len(selected) < total_events:
            raise ValueError(
                "Not enough independent events found to meet the specified number of events per year."
            )

        threshold_value = min(selected_values)

        thresholds.append({
            "Threshold": threshold_value,
            "Number_of_Years": total_years,
            "Target_Total_Events": total_events
        })

        for t, val in zip(selected, selected_values):

            max_hourly= get_hourly_associated_max(
                df_associated_hourly_variable,
                associated_hourly_variable_input_column,
                t,
                associated_variable_decluster_window
            )


            all_pot.append({
                "time": t,
                center_variable_output_column: val,
                associated_hourly_variable_output_column: max_hourly
            })

    # ─────────────────────────────────────────────────────────────
    # Non-stationary POT threshold
    # ─────────────────────────────────────────────────────────────
    else:

        for year, group in center_variable_series.groupby(center_variable_series.index.year):

            group_sorted = group.sort_values(ascending=False)

            selected = []
            selected_values = []

            for idx, val in group_sorted.items():

                if any(abs(idx - sel) <= decluster_window for sel in selected):
                    continue

                selected.append(idx)
                selected_values.append(val)

                if len(selected) == number_of_events_per_year:
                    break

            if len(selected) < number_of_events_per_year:
                print(f"Skipping {year}: not enough independent events.")
                continue

            threshold_value = min(selected_values)

            thresholds.append({
                "Year": year,
                "Threshold": threshold_value
            })

            for t, val in zip(selected, selected_values):

                max_hourly = get_hourly_associated_max(
                    df_associated_hourly_variable,
                    associated_hourly_variable_input_column,
                    t,
                    associated_variable_decluster_window
                )

                all_pot.append({
                    "time": t,
                    center_variable_output_column: val,
                    associated_hourly_variable_output_column: max_hourly
                })

    # ─────────────────────────────────────────────────────────────
    # Save outputs
    # ─────────────────────────────────────────────────────────────
    merged_df = pd.DataFrame(all_pot).sort_values("time").reset_index(drop=True)

    df_thresh = pd.DataFrame(thresholds)
    df_thresh.to_csv(thresholds_output_path, index=False)

    # merged_df = merged_df.dropna(
    #     subset=[
    #         center_variable_output_column,
    #         associated_hourly_variable_output_column,
    #         associated_second_hourly_variable_output_column
    #     ]
    # )

    merged_df["time"] = pd.to_datetime(merged_df["time"])
    merged_df.to_csv(pot_output_path, index=False)

    return merged_df, df_thresh

In [ ]:
n_events = 4
POT_Hourly_all('time','time','Storm_Surge','Rainfall',
           r'path/to/center_variable.csv',
           r'path/to/associated_variable.csv',
           'POT_NTR','Max_RF', n_events,
           rf'path/to/Thresholds_NTR_{n_events}.csv',
           rf'path/to/POT_NTR_Functions_{n_events}.csv',
          stationary=False
          )